# Automatic segmentation of bumps in PSD of EEG/LFP or OU simulated signals.


We generate a signal that is the sum of OU process to simulate some EEG/ LFP signal. Since we generated the signal we know exact parameter. In normal condition the signal come from real physiology and we would like to estimate the OU process which sum leads to the final signal PSD. The goal of doing so is to use the exact formula of the OU process on an unknown signal to automatically segment its bumps and decay area. 

Here we consequently check the last part of segmentation of bump and decay is feasible once a set of OU processes is fitted. Except instead of fitting OU process we directly use the OU set exact parameters.

In [28]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import scipy as sc
import scipy.signal as signal
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals_exact, get_analytical_psd
from Functions.fit_OU import fit_ou_mixture_psd, multi_ou_psd
from Functions.fit_student_t import fit_student_t_mixture_psd, multi_student_t_psd, fit_dual_student_t_mixture_psd, multi_dual_student_t_psd
from Functions.fit_pearson import fit_pearson_iv_mixture_psd, multi_pearson_iv_psd
from Functions.fit_pseudo_voigt import fit_pseudo_voigt_mixture_psd, multi_pseudo_voigt_psd
from Functions.time_frequency import spectrogram
from specparam import SpectralModel # for FOOOF method
from scipy.optimize import least_squares
from Functions.smooth import smooth_savitzky_golay, smooth_adaptive_whittaker, smooth_log_octave

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Generate synthetic signal

In [146]:
# --- Parameters
T = 1000 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 10]
factor_list = [1, 1, 0.01]

# --- Generate EEG data
t, y = get_mixed_OU_signals_exact(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')

axes[1].sharex(axes[0])


plt.show()

# 2. Segmentation of signal bump and decay areas using OU set parameters

As explained in introduction we do not try to fit set of OU to the data but directly use the exact parameters to check the feasibiliy of this segmentation.

In [220]:
f, psd_a = get_analytical_psd(200, 40, lbda_list, omega_list, sigma_list, factor_list)

f_welch, psd_welch = sc.signal.welch(y, fs, nperseg = 16 *fs, average='median')
mask = f_welch <= 40


fig, axes = plt.subplots(3, constrained_layout = True)
axes[0].plot(f_spectro, np.mean(spectro, axis = 1))
axes[0].plot(f, psd_a)
axes[1].semilogy(f_spectro, np.mean(spectro, axis = 1))
axes[1].semilogy(f_spectro, np.median(spectro, axis = 1))
axes[1].semilogy(f, psd_a)
axes[2].semilogy(f_welch[mask], psd_welch[mask])
axes[2].semilogy(f, psd_a)

plt.show()

__OBSERVATIONS:__

Not adding the division factor of pi and keeping the multiplicative factor of 2 to get analytical psd sum seems to work when psd is computed by welch and high nperseg.


## Fit functions of the PSD using multiple OU 

__Example on OU simulated signal__

On this example we should reconstruct a PSD similar to the one made above with the analytical curve

In [130]:
# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_welch > 0) & (f_welch < 45.0)
f_fit = f_welch[fit_mask]
psd_fit = psd_welch[fit_mask]

# Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_ou_mixture_psd(f_fit, psd_fit, max_components=5)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components: # display peaks components
    print(comp)

#  Plot Empirical vs Fitted PSD
psd_model = multi_ou_psd(f_fit, *popt)

# --- Log-domain Savitzky-Golay Smoothing ---
log_psd_fit = np.log(psd_fit)
log_psd_smoothed = sc.signal.savgol_filter(log_psd_fit, window_length=5, polyorder=1)
psd_smoothed = np.exp(log_psd_smoothed)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_smoothed, 'b-', alpha=0.6, label='Empirical Welch PSD smoothed')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed OU Processes')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 5
{'component': 1, 'A_est (c*sigma)': 7.384511192893789, 'lambda_est': 0.9678925033709259, 'f0_est_Hz': 0.9970020262177842}
{'component': 2, 'A_est (c*sigma)': 5.102152951852055, 'lambda_est': 2.109171100360724, 'f0_est_Hz': 9.984928194032596}
{'component': 3, 'A_est (c*sigma)': 0.08090908986119623, 'lambda_est': 2.339606200493337, 'f0_est_Hz': 27.517930778374826}
{'component': 4, 'A_est (c*sigma)': 0.2446629016665968, 'lambda_est': 1.0127814170956104, 'f0_est_Hz': 30.008726928622355}
{'component': 5, 'A_est (c*sigma)': 0.008318292510212249, 'lambda_est': 0.1770299385996957, 'f0_est_Hz': 39.29192106114467}


__Example on real EEG during GA__

__Lorentzian mixture fit__

In [222]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 2))

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_ou_mixture_psd(f_fit, psd_fit, prominence = 1, max_components=4)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components:
    print(comp)

# 4. Plot Empirical vs Fitted PSD
psd_model = multi_ou_psd(f_fit, *popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed OU Processes')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 4
{'component': 1, 'A_est (c*sigma)': 40.30802689753902, 'lambda_est': 0.0010000002745798008, 'f0_est_Hz': 0.7394265045013694}
{'component': 2, 'A_est (c*sigma)': 36.11408172154405, 'lambda_est': 5.851919910137115, 'f0_est_Hz': 12.439225353298827}
{'component': 3, 'A_est (c*sigma)': 23.96642329313774, 'lambda_est': 6.523738589758375, 'f0_est_Hz': 5.858634483561942}
{'component': 4, 'A_est (c*sigma)': 8.147998779751712e-05, 'lambda_est': 99.4664371896321, 'f0_est_Hz': 0.32715176541932156}


__Comparison with FOOOF method__

In [184]:
freq_range = [0.1, 45]
fm = SpectralModel(aperiodic_mode='fixed', periodic_mode='cauchy', peak_width_limits=[0.2, 15.0], max_n_peaks=3, min_peak_height=0.1)
fm.fit(f_emp, psd_emp, freq_range)
fm.report(f_emp, psd_emp, freq_range)

                                                                                                  
                                      SPECTRUM MODEL RESULTS                                      
                                                                                                  
                       The model was fit with the 'spectral_fit' algorithm                        
               Model was fit to the 0-45 Hz frequency range with 0.06 Hz resolution               
                                                                                                  
                               Aperiodic Parameters ('fixed' mode)                                
                                        (offset, exponent)                                        
                                          1.9151, 2.2126                                          
                                                                                                  
          

## Fit of PSD using multiple Student-t

In [25]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 16)) #f_welch, psd_welch

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_student_t_mixture_psd(f_fit, psd_fit, prominence=1, max_components=5)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components:
    print(comp)

# 4. Plot Empirical vs Fitted PSD
psd_model = multi_student_t_psd(f_fit, *popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed Student-t')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 4
{'component': 1, 'A_est': 37.51417924272938, 'gamma_est (width)': 0.31145152504096896, 'f0_est_Hz': 0.56941425477061, 'alpha_est (tail exponent)': 2.460850653184636}
{'component': 2, 'A_est': 5.346618248496164, 'gamma_est (width)': 1.6846731656339118, 'f0_est_Hz': 12.492438382514134, 'alpha_est (tail exponent)': 4.188073044189938}
{'component': 3, 'A_est': 2.843497991288979, 'gamma_est (width)': 14.394338978470552, 'f0_est_Hz': 3.024551306988186, 'alpha_est (tail exponent)': 5.999998267630504}
{'component': 4, 'A_est': 0.6759578900288474, 'gamma_est (width)': 8.50173225008644, 'f0_est_Hz': 20.8517529201257, 'alpha_est (tail exponent)': 4.074381057647089}


## Fit of PSD using Multiple dual Student-t model

In [29]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 16)) #f_welch, psd_welch

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_dual_student_t_mixture_psd(f_fit, psd_fit, prominence=1, max_components=4)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components:
    print(comp)

# 4. Plot Empirical vs Fitted PSD
psd_model = multi_dual_student_t_psd(f_fit, *popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed OU Processes')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 3
{'component': 1, 'A_sharp_est': 39.20741901935949, 'gamma_sharp_est': 0.27934811361475936, 'alpha_sharp_est': 2.4478514999663528, 'A_broad_est': 3.2622443737294895, 'gamma_broad_est': 9.464477755128609, 'alpha_broad_est': 5.999987184702262, 'f0_est_Hz': 0.569407957998749}
{'component': 2, 'A_sharp_est': 5.315749325723402, 'gamma_sharp_est': 1.4222307741556652, 'alpha_sharp_est': 5.160732601532574, 'A_broad_est': 1.3951381434033348, 'gamma_broad_est': 16.349260587667864, 'alpha_broad_est': 5.999999999999999, 'f0_est_Hz': 12.744060964010673}
{'component': 3, 'A_sharp_est': 2.2868479892469447, 'gamma_sharp_est': 0.4881744953466179, 'alpha_sharp_est': 5.993877482515892, 'A_broad_est': 2.406884847399832, 'gamma_broad_est': 4.233896406057307, 'alpha_broad_est': 5.999999813428113, 'f0_est_Hz': 11.008774095823638}


## Fit PSd using Multiple Pearson

In [26]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 16)) #f_welch, psd_welch

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_pearson_iv_mixture_psd(f_fit, psd_fit, prominence=1, max_components=5)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components:
    print(comp)

# 4. Plot Empirical vs Fitted PSD
psd_model = multi_pearson_iv_psd(f_fit, *popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed Pearson')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 4
{'component': 1, 'A_est': 22.04127626031328, 'gamma_est (width)': 0.23654463244963841, 'f0_est_Hz': 0.486012510109182, 'm_est (tail exponent)': 1.2451997408640456, 'nu_p_est (asymmetry)': -0.5750282296430975}
{'component': 2, 'A_est': 2.8136208673566268, 'gamma_est (width)': 1.3294090609726246, 'f0_est_Hz': 13.125420783093919, 'm_est (tail exponent)': 1.559874285122216, 'nu_p_est (asymmetry)': 0.9745039552298985}
{'component': 3, 'A_est': 1.6232802350944109, 'gamma_est (width)': 51.07768130883189, 'f0_est_Hz': 0.1913775602292127, 'm_est (tail exponent)': 5.903746436467381, 'nu_p_est (asymmetry)': 4.999999999991046}
{'component': 4, 'A_est': 0.009333717820545451, 'gamma_est (width)': 0.14834336402181578, 'f0_est_Hz': 26.98108197430211, 'm_est (tail exponent)': 0.20000000028012452, 'nu_p_est (asymmetry)': 4.8777953486620484}


## Fit PSD using Multiple Pseudo-Voigt

In [27]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 16)) #f_welch, psd_welch

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Automatic Inverse Fit
best_K, fitted_components, popt = fit_pseudo_voigt_mixture_psd(f_fit, psd_fit, prominence=1, max_components=5)

print(f"--- INVERSE FIT RESULTS ---")
print(f"Estimated number of OU processes (K): {best_K}")
for comp in fitted_components:
    print(comp)

# 4. Plot Empirical vs Fitted PSD
psd_model = multi_pseudo_voigt_psd(f_fit, *popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K})')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fitting for Mixed Pseudo-Voigt')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- INVERSE FIT RESULTS ---
Estimated number of OU processes (K): 4
{'component': 1, 'A_est': 17.310442478669003, 'gamma_est (width)': 0.6096515455557238, 'f0_est_Hz': 0.3748263440773942, 'eta_est (mixing factor)': 0.31151404821096573}
{'component': 2, 'A_est': 4.708565205954312, 'gamma_est (width)': 12.397598254995309, 'f0_est_Hz': 1.3794252854528957, 'eta_est (mixing factor)': 8.923624783890802e-37}
{'component': 3, 'A_est': 5.0820219182041075, 'gamma_est (width)': 0.09766992247792734, 'f0_est_Hz': 5.287616672843853, 'eta_est (mixing factor)': 6.111599907178331e-36}
{'component': 4, 'A_est': 5.159856827956642, 'gamma_est (width)': 0.26698407250734657, 'f0_est_Hz': 13.421654982406132, 'eta_est (mixing factor)': 1.5973627604229513e-41}


__Visualize different components__

In [236]:
# Compute the total fitted model
psd_model = multi_student_t_psd(f_fit, *popt)

# Extract parameters per component
# Assuming each Student-t component has N parameters (e.g., A, gamma, f0, alpha)
num_params_per_comp = len(popt) // best_K

plt.figure(figsize=(10, 6))

# 1. Plot raw Welch PSD
plt.semilogy(f_fit, psd_fit, color='black', alpha=0.4, linewidth=1.5, label='Empirical Welch PSD')

# 2. Plot individual Student-t components
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

for k in range(best_K):
    # Slice parameters for the k-th component
    comp_params = popt[k * num_params_per_comp : (k + 1) * num_params_per_comp]
    
    # If you have a single-component PSD function:
    # psd_comp = single_student_t_psd(f_fit, *comp_params)
    #
    # Otherwise, evaluate multi_student_t_psd with K=1 equivalent:
    psd_comp = multi_student_t_psd(f_fit, *comp_params)
    
    # Get peak center for label context
    f0 = fitted_components[k].get('f0_est_Hz', 0.0)
    color = colors[k % len(colors)]
    
    plt.semilogy(
        f_fit, 
        psd_comp, 
        linestyle='--', 
        linewidth=1.5, 
        color=color, 
        label=f'Comp {k+1} ($f_0 \\approx {f0:.1f}$ Hz)'
    )

# 3. Plot total combined model fit
plt.semilogy(f_fit, psd_model, 'r-', linewidth=2.5, label=f'Total Model (K={best_K})')

# Formatting
plt.xlabel('Frequency [Hz]', fontsize=11)
plt.ylabel('PSD [Power/Hz]', fontsize=11)
plt.title('Inverse PSD Fitting: Total Model & Individual Student-t Components', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

__Extract peaks frequency ranges based on fitted component peak widths__

In [234]:
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Extraction Parameters ---
MIN_PEAK_RATIO = 0.5  # Peak must reach >= 50% of the empirical PSD height
MAX_FWHM_HZ = 20.0    # Maximum allowed FWHM in Hz

num_params_per_comp = len(popt) // best_K
psd_model = multi_student_t_psd(f_fit, *popt)

extracted_intervals = []

# --- 2. Filter & Extract Half-Width Intervals Directly from Components ---
for k in range(best_K):
    comp_params = popt[k * num_params_per_comp : (k + 1) * num_params_per_comp]
    psd_comp = multi_student_t_psd(f_fit, *comp_params)
    
    # Peak amplitude and center
    peak_idx = np.argmax(psd_comp)
    comp_peak_val = psd_comp[peak_idx]
    empirical_val_at_f0 = psd_fit[peak_idx]
    
    # 1. Height filter: reject if too low relative to empirical PSD
    if comp_peak_val < (MIN_PEAK_RATIO * empirical_val_at_f0):
        continue
        
    # 2. Extract half-width indices directly on the component
    half_max_val = comp_peak_val / 2.0
    above_half_max = np.where(psd_comp >= half_max_val)[0]
    
    if len(above_half_max) < 2:
        continue
        
    f_left = f_fit[above_half_max[0]]
    f_right = f_fit[above_half_max[-1]]
    fwhm_hz = f_right - f_left
    
    # 3. Width filter: reject overly broad components
    if fwhm_hz > MAX_FWHM_HZ:
        continue
        
    extracted_intervals.append({
        'comp_id': k + 1,
        'f0': f_fit[peak_idx],
        'f_left': f_left,
        'f_right': f_right,
        'fwhm_hz': fwhm_hz,
        'half_max_val': half_max_val,
        'psd_comp': psd_comp
    })

# --- Print Results ---
print("Extracted Direct Component Half-Width Intervals:")
for res in extracted_intervals:
    print(f"Component {res['comp_id']}: [{res['f_left']:.2f} Hz - {res['f_right']:.2f} Hz] (FWHM = {res['fwhm_hz']:.2f} Hz)")

# --- 3. Plotting ---
plt.figure(figsize=(10, 6))

# Empirical & Total Fit
plt.semilogy(f_fit, psd_fit, color='black', alpha=0.3, linewidth=1.5, label='Empirical Welch PSD')
plt.semilogy(f_fit, psd_model, 'r-', linewidth=2.0, label=f'Total Fit (K={best_K})')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Plot kept components and mark their half-width interval directly
for res in extracted_intervals:
    k = res['comp_id'] - 1
    color = colors[k % len(colors)]
    
    # Plot component curve
    plt.semilogy(
        f_fit, 
        res['psd_comp'], 
        linestyle='--', 
        linewidth=1.5, 
        color=color, 
        label=f"Comp {res['comp_id']} ($f_0 \\approx {res['f0']:.1f}$ Hz)"
    )
    
    # Horizontal line across the half-width interval directly on the component
    plt.hlines(
        y=res['half_max_val'], 
        xmin=res['f_left'], 
        xmax=res['f_right'], 
        color=color, 
        linewidth=2.5
    )
    
    # Vertical boundary markers
    plt.vlines(
        x=[res['f_left'], res['f_right']], 
        ymin=res['half_max_val'] * 0.85, 
        ymax=res['half_max_val'] * 1.15, 
        color=color, 
        linewidth=1.5
    )

plt.xlabel('Frequency [Hz]', fontsize=11)
plt.ylabel('PSD [Power/Hz]', fontsize=11)
plt.title('Direct Half-Width Intervals of Valid Peaks', fontsize=12)
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend(loc='upper right', framealpha=0.9)
plt.tight_layout()
plt.show()

Extracted Direct Component Half-Width Intervals:
Component 1: [0.31 Hz - 0.81 Hz] (FWHM = 0.50 Hz)
Component 2: [11.44 Hz - 13.50 Hz] (FWHM = 2.06 Hz)


__Peak detection Student-t fitted PSD__

In [229]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_prominences
from scipy.sparse import csc_matrix, diags
from scipy.sparse.linalg import spsolve

# ==============================================================================
# 1. WHITTAKER SMOOTHER FUNCTIONS
# ==============================================================================

def whittaker_smooth_baseline(y, lam=1e5, p=0.005, niter=10):
    """Fits a smooth global continuum baseline to log10(PSD)."""
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L))
    D = csc_matrix(D)
    w = np.ones(L)
    for _ in range(niter):
        W = diags([w], [0], shape=(L, L))
        Z = W + lam * D.T * D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

def whittaker_smooth_segment(y, lam=1e4, p=0.01, niter=10):
    """Fits an asymmetric Whittaker baseline to a smaller segment y."""
    L = len(y)
    if L < 4:
        return np.copy(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L))
    D = csc_matrix(D)
    w = np.ones(L)
    for _ in range(niter):
        W = diags([w], [0], shape=(L, L))
        Z = W + lam * D.T * D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# ==============================================================================
# 2. PROMINENCE BASE EXTRACTION (3 CONDITIONS)
# ==============================================================================

def extract_prominence_bases_3way_comparison(f_fit, psd_fit, psd_model, popt, best_K,
                                             min_peak_ratio=0.5, max_fwhm_hz=20.0):
    num_params_per_comp = len(popt) // best_K
    log_model = np.log10(psd_model)

    # Helper function to match detected peaks with valid fitted components
    def match_and_filter_peaks(peaks, left_bases, right_bases, f_grid):
        matched = []
        for idx, p_idx in enumerate(peaks):
            f0_peak = f_grid[p_idx]
            
            for k in range(best_K):
                comp_params = popt[k * num_params_per_comp : (k + 1) * num_params_per_comp]
                psd_comp = multi_student_t_psd(f_fit, *comp_params)
                comp_peak_idx = np.argmax(psd_comp)

                if np.abs(f_fit[comp_peak_idx] - f0_peak) < 1.0:
                    empirical_val = psd_fit[np.argmin(np.abs(f_fit - f0_peak))]
                    if psd_comp[comp_peak_idx] >= (min_peak_ratio * empirical_val):
                        half_max = psd_comp[comp_peak_idx] / 2.0
                        above_hm = np.where(psd_comp >= half_max)[0]
                        if len(above_hm) >= 2 and (f_fit[above_hm[-1]] - f_fit[above_hm[0]]) <= max_fwhm_hz:
                            l_idx = left_bases[idx]
                            r_idx = right_bases[idx]
                            matched.append({
                                'comp_id': k + 1,
                                'f0': f0_peak,
                                'l_idx': l_idx,
                                'r_idx': r_idx,
                                'f_L': f_grid[l_idx],
                                'f_R': f_grid[r_idx],
                                'width_hz': f_grid[r_idx] - f_grid[l_idx]
                            })
                            break
        return matched

    # --------------------------------------------------------------------------
    # CONDITION 1: DIRECT FIT
    # --------------------------------------------------------------------------
    peaks_fit, _ = find_peaks(log_model, distance=3, prominence=0.01)
    _, left_fit, right_fit = peak_prominences(log_model, peaks_fit)
    matched_direct = match_and_filter_peaks(peaks_fit, left_fit, right_fit, f_fit)

    # --------------------------------------------------------------------------
    # CONDITION 2: GLOBAL BASELINE SUBTRACTION
    # --------------------------------------------------------------------------
    log_base_global = whittaker_smooth_baseline(log_model, lam=1e5, p=0.005)
    psd_base_global = 10**log_base_global
    excess_global = np.maximum(0, log_model - log_base_global)

    peaks_glob, _ = find_peaks(excess_global, distance=3, prominence=0.01)
    _, left_glob, right_glob = peak_prominences(excess_global, peaks_glob)
    matched_global = match_and_filter_peaks(peaks_glob, left_glob, right_glob, f_fit)

    # --------------------------------------------------------------------------
    # CONDITION 3: LOCAL SEGMENTED BASELINE SUBTRACTION
    # --------------------------------------------------------------------------
    valid_peaks_info = sorted(matched_global, key=lambda x: x['f0'])
    num_valid = len(valid_peaks_info)

    matched_local = []
    local_segment_data = []

    for i, peak in enumerate(valid_peaks_info):
        f0 = peak['f0']

        # Determine Left/Right Midpoint Boundaries
        left_mid_f = f_fit[0] if i == 0 else (valid_peaks_info[i - 1]['f0'] + f0) / 2.0
        right_mid_f = f_fit[-1] if i == num_valid - 1 else (f0 + valid_peaks_info[i + 1]['f0']) / 2.0

        idx_start = np.argmin(np.abs(f_fit - left_mid_f))
        idx_end = np.argmin(np.abs(f_fit - right_mid_f))

        f_segment = f_fit[idx_start : idx_end + 1]
        log_segment = log_model[idx_start : idx_end + 1]

        # Local Whittaker baseline per segment
        log_base_seg = whittaker_smooth_segment(log_segment, lam=1e4, p=0.01)
        excess_seg = np.maximum(0, log_segment - log_base_seg)

        # Find peak on local segment
        p_seg, _ = find_peaks(excess_seg, distance=3, prominence=0.01)
        if len(p_seg) > 0:
            _, l_seg, r_seg = peak_prominences(excess_seg, p_seg)
            # Pick peak closest to f0
            best_p = p_seg[np.argmin(np.abs(f_segment[p_seg] - f0))]
            best_p_idx = np.where(p_seg == best_p)[0][0]

            l_idx_local = l_seg[best_p_idx]
            r_idx_local = r_seg[best_p_idx]

            matched_local.append({
                'comp_id': peak['comp_id'],
                'f0': f0,
                'f_L': f_segment[l_idx_local],
                'f_R': f_segment[r_idx_local],
                'width_hz': f_segment[r_idx_local] - f_segment[l_idx_local]
            })

            local_segment_data.append({
                'f_segment': f_segment,
                'excess_seg': excess_seg,
                'f_L': f_segment[l_idx_local],
                'f_R': f_segment[r_idx_local]
            })

    return matched_direct, matched_global, matched_local, psd_base_global, excess_global, local_segment_data

# ==============================================================================
# 3. RUN COMPARISON & PLOT
# ==============================================================================

psd_model = multi_student_t_psd(f_fit, *popt)

direct_res, global_res, local_res, psd_base_global, excess_global, local_seg_data = extract_prominence_bases_3way_comparison(
    f_fit, psd_fit, psd_model, popt, best_K, 
    min_peak_ratio=MIN_PEAK_RATIO, 
    max_fwhm_hz=MAX_FWHM_HZ
)

# Print Summary Table
print("\n=========================================================================================================================")
print(f"{'Comp':<5} | {'f0 (Hz)':<8} | {'Direct Fit Bounds':<25} | {'Global Baseline Bounds':<25} | {'Local Segmented Bounds':<25}")
print("=========================================================================================================================")
for d, g, l in zip(direct_res, global_res, local_res):
    print(f"#{d['comp_id']:<4} | {d['f0']:<8.2f} | "
          f"[{d['f_L']:5.2f} - {d['f_R']:5.2f}] ({d['width_hz']:4.2f} Hz)   | "
          f"[{g['f_L']:5.2f} - {g['f_R']:5.2f}] ({g['width_hz']:4.2f} Hz)   | "
          f"[{l['f_L']:5.2f} - {l['f_R']:5.2f}] ({l['width_hz']:4.2f} Hz)")
print("=========================================================================================================================\n")

# --- 3-Panel Visual Comparison Plot ---
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# --- Panel 1: Direct Fit ---
ax1.semilogy(f_fit, psd_fit, color='gray', alpha=0.35, linewidth=1.2, label='Empirical Welch PSD')
ax1.semilogy(f_fit, psd_model, 'r-', linewidth=2.2, label=f'Total Fit (K={best_K})')

for i, res in enumerate(direct_res):
    color = colors[i % len(colors)]
    mask = (f_fit >= res['f_L']) & (f_fit <= res['f_R'])
    base_val = min(psd_model[res['l_idx']], psd_model[res['r_idx']])
    
    ax1.fill_between(f_fit[mask], psd_model[mask], y2=base_val, color=color, alpha=0.3,
                     label=f"Bump {res['comp_id']} Direct [{res['f_L']:.1f}-{res['f_R']:.1f} Hz]")
    ax1.plot([res['f_L'], res['f_R']], [psd_model[res['l_idx']], psd_model[res['r_idx']]], 
             'D', color=color, markersize=7, markeredgecolor='black')

ax1.set_ylabel('PSD [Power/Hz]', fontsize=10)
ax1.set_title('Condition 1: Prominence Bases Calculated Directly on Total Fit', fontsize=11)
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc='upper right', framealpha=0.9, fontsize=8)

# --- Panel 2: Global Baseline Subtraction ---
ax2.plot(f_fit, excess_global, 'k-', linewidth=1.5, label='Global Excess Log Power')

for i, res in enumerate(global_res):
    color = colors[i % len(colors)]
    mask = (f_fit >= res['f_L']) & (f_fit <= res['f_R'])
    
    ax2.fill_between(f_fit[mask], excess_global[mask], y2=0, color=color, alpha=0.35,
                     label=f"Bump {res['comp_id']} Global [{res['f_L']:.1f}-{res['f_R']:.1f} Hz]")
    ax2.plot([res['f_L'], res['f_R']], [excess_global[res['l_idx']], excess_global[res['r_idx']]], 
             'o', color=color, markersize=7, markeredgecolor='black')

ax2.set_ylabel('Excess Log Power', fontsize=10)
ax2.set_title('Condition 2: Prominence Bases Calculated on Single Global Baseline Residual', fontsize=11)
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend(loc='upper right', framealpha=0.9, fontsize=8)

# --- Panel 3: Local Segmented Baseline Subtraction ---
ax3.axhline(0, color='black', linestyle='--', alpha=0.5)

for i, seg in enumerate(local_seg_data):
    color = colors[i % len(colors)]
    
    ax3.plot(seg['f_segment'], seg['excess_seg'], color=color, linewidth=1.8, 
             label=f"Local Diff Bump {i+1}")
    
    mask = (seg['f_segment'] >= seg['f_L']) & (seg['f_segment'] <= seg['f_R'])
    ax3.fill_between(seg['f_segment'][mask], seg['excess_seg'][mask], y2=0, color=color, alpha=0.35)
    
    idx_L = np.argmin(np.abs(seg['f_segment'] - seg['f_L']))
    idx_R = np.argmin(np.abs(seg['f_segment'] - seg['f_R']))
    ax3.plot([seg['f_L'], seg['f_R']], [seg['excess_seg'][idx_L], seg['excess_seg'][idx_R]], 
             's', color=color, markersize=7, markeredgecolor='black')

ax3.set_xlabel('Frequency [Hz]', fontsize=10)
ax3.set_ylabel('Local Excess Power', fontsize=10)
ax3.set_title('Condition 3: Prominence Bases Calculated on Local Segmented Baselines', fontsize=11)
ax3.grid(True, ls="--", alpha=0.5)
ax3.legend(loc='upper right', framealpha=0.9, fontsize=8)

plt.tight_layout()
plt.show()


Comp  | f0 (Hz)  | Direct Fit Bounds         | Global Baseline Bounds    | Local Segmented Bounds   
#1    | 0.56     | [ 0.06 - 44.94] (44.88 Hz)   | [ 0.06 -  3.44] (3.38 Hz)   | [ 0.06 -  2.12] (2.06 Hz)
#2    | 12.50    | [ 9.31 - 44.94] (35.62 Hz)   | [ 8.44 - 16.94] (8.50 Hz)   | [10.06 - 15.56] (5.50 Hz)



__peak detection raw psd__

In [230]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_prominences
from scipy.ndimage import gaussian_filter1d
from scipy.sparse import csc_matrix, diags
from scipy.sparse.linalg import spsolve

# 1. Whittaker Baseline
def whittaker_smooth_baseline(y, lam=1e5, p=0.005, niter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L))
    D = csc_matrix(D)
    w = np.ones(L)
    for _ in range(niter):
        W = diags([w], [0], shape=(L, L))
        Z = W + lam * D.T * D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

def extract_tuned_non_parametric_bumps(f_fit, psd_emp, 
                                        min_dist_hz=1.5,       # Minimum distance between peaks in Hz
                                        min_prominence=0.3,    # Minimum prominence in log units
                                        min_width_hz=0.5,      # Minimum peak width in Hz
                                        smooth_sigma_hz=0.3):  # Light Gaussian smoothing sigma in Hz
    df = f_fit[1] - f_fit[0]
    log_emp = np.log10(psd_emp)
    
    # 1. Baseline estimation
    log_bg = whittaker_smooth_baseline(log_emp, lam=1e5, p=0.005)
    psd_bg = 10**log_bg
    
    # 2. Excess Log Power
    excess_raw = np.maximum(0, log_emp - log_bg)
    
    # 3. Light Non-Parametric Gaussian Smoothing to remove noise ripples
    sigma_pts = smooth_sigma_hz / df
    excess_smoothed = gaussian_filter1d(excess_raw, sigma=sigma_pts)

    # 4. Convert Hz thresholds into array index counts
    distance_pts = int(min_dist_hz / df)
    width_pts = int(min_width_hz / df)

    # 5. FIND PEAKS WITH TUNED PARAMETERS
    peaks, _ = find_peaks(
        excess_smoothed, 
        distance=distance_pts, 
        prominence=min_prominence,
        width=width_pts
    )
    
    prominences, left_bases, right_bases = peak_prominences(excess_smoothed, peaks)

    extracted_bumps = []
    for i, p_idx in enumerate(peaks):
        l_idx = left_bases[i]
        r_idx = right_bases[i]
        
        extracted_bumps.append({
            'bump_id': i + 1,
            'f0': f_fit[p_idx],
            'f_L': f_fit[l_idx],
            'f_R': f_fit[r_idx],
            'width_hz': f_fit[r_idx] - f_fit[l_idx],
            'p_idx': p_idx,
            'l_idx': l_idx,
            'r_idx': r_idx
        })

    return extracted_bumps, psd_bg, excess_raw, excess_smoothed

# Run with tuned parameters
bumps, psd_bg, excess_raw, excess_smoothed = extract_tuned_non_parametric_bumps(
    f_fit, psd_fit, 
    min_dist_hz=1.5,     # Require peaks to be at least 1.5 Hz apart
    min_prominence=0.3,  # Ignore minor noise bumps under 0.3 log units
    min_width_hz=0.5,    # Ignore sharp 1-sample spikes
    smooth_sigma_hz=0.3  # Smooth out high-frequency noise ripples
)

print("\n=========================================================================================")
print(f"{'Bump':<5} | {'Center f0 (Hz)':<15} | {'Base Frequency Interval':<30} | {'Bandwidth (Hz)':<15}")
print("=========================================================================================")
for b in bumps:
    print(f"#{b['bump_id']:<4} | {b['f0']:<15.2f} | "
          f"[{b['f_L']:5.2f} Hz  -  {b['f_R']:5.2f} Hz]                 | "
          f"{b['width_hz']:4.2f} Hz")
print("=========================================================================================\n")

# ==============================================================================
# 4. TWO-PANEL VISUAL DISPLAY
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# --- Panel 1: Empirical PSD, Baseline, and Extracted Bump Ranges ---
ax1.semilogy(f_fit, psd_fit, color='gray', alpha=0.5, linewidth=1.2, label='Empirical Welch PSD')
ax1.semilogy(f_fit, psd_bg, 'k--', linewidth=2.0, label='Aperiodic Baseline ($1/f^{\\alpha}$ Background)')

for i, b in enumerate(bumps):
    color = colors[i % len(colors)]
    
    # Shade peak region down to baseline
    mask = (f_fit >= b['f_L']) & (f_fit <= b['f_R'])
    ax1.fill_between(
        f_fit[mask], psd_fit[mask], y2=psd_bg[mask],
        color=color, alpha=0.35,
        label=f"Bump {b['bump_id']} [{b['f_L']:.1f} - {b['f_R']:.1f} Hz]"
    )
    
    # Diamond boundary markers on the baseline
    ax1.plot([b['f_L'], b['f_R']], [psd_bg[b['l_idx']], psd_bg[b['r_idx']]], 
             'D', color=color, markersize=7, markeredgecolor='black')
    
    # Peak center marker
    ax1.plot(b['f0'], psd_fit[b['p_idx']], 'o', color=color, markersize=8, markeredgecolor='black')

ax1.set_ylabel('PSD [Power/Hz]', fontsize=11)
ax1.set_title('Non-Parametric Peak & Boundary Detection (No Student\'s t Model Fit Needed)', fontsize=12)
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc='upper right', framealpha=0.9, fontsize=9)

# --- Panel 2: Baseline-Subtracted Excess Log Power & Topological Bases ---
# FIX: Use 'excess_raw' (or 'excess_smoothed') returned by the function
ax2.plot(f_fit, excess_raw, 'k-', linewidth=1.5, label='Raw Excess Log Power ($log_{10} PSD - log_{10} Baseline$)')
ax2.plot(f_fit, excess_smoothed, 'k-', linewidth=1.5, label='Raw Excess Log Power ($log_{10} PSD - log_{10} Baseline$)')

for i, b in enumerate(bumps):
    color = colors[i % len(colors)]
    
    # Shade excess power region
    mask = (f_fit >= b['f_L']) & (f_fit <= b['f_R'])
    ax2.fill_between(f_fit[mask], excess_raw[mask], y2=0, color=color, alpha=0.35)
    
    # Circle markers for left/right bases on flattened background
    ax2.plot([b['f_L'], b['f_R']], [excess_raw[b['l_idx']], excess_raw[b['r_idx']]], 
             'o', color=color, markersize=7, markeredgecolor='black')

ax2.set_xlabel('Frequency [Hz]', fontsize=11)
ax2.set_ylabel('Excess Log Power', fontsize=11)
ax2.set_title('Detrended Flat Spectrum with Prominence Bases', fontsize=12)
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend(loc='upper right', framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.show()


Bump  | Center f0 (Hz)  | Base Frequency Interval        | Bandwidth (Hz) 
#1    | 12.62           | [ 4.88 Hz  -  17.19 Hz]                 | 12.31 Hz



In [205]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_prominences
from scipy.ndimage import gaussian_filter1d
from scipy.sparse import csc_matrix, diags
from scipy.sparse.linalg import spsolve

# ==============================================================================
# 1. WHITTAKER CONTINUUM BASELINE
# ==============================================================================

def whittaker_smooth_baseline(y, lam=1e5, p=0.005, niter=10):
    L = len(y)
    D = diags([1, -2, 1], [0, 1, 2], shape=(L - 2, L))
    D = csc_matrix(D)
    w = np.ones(L)
    for _ in range(niter):
        W = diags([w], [0], shape=(L, L))
        Z = W + lam * D.T * D
        z = spsolve(Z, w * y)
        w = p * (y > z) + (1 - p) * (y <= z)
    return z

# ==============================================================================
# 2. NON-PARAMETRIC ALGORITHM APPLIED DIRECTLY TO RAW (NON-SMOOTHED) RESIDUAL
# ==============================================================================

def extract_non_parametric_raw(f_fit, psd_emp, 
                               min_dist_hz=1.5,       # Minimum distance between peaks in Hz
                               min_prominence=0.3,    # Minimum prominence in log units
                               min_width_hz=0.5,      # Minimum peak width in Hz
                               smooth_sigma_hz=0.3):  # For visual reference only
    df = f_fit[1] - f_fit[0]
    log_emp = np.log10(psd_emp)
    
    # 1. Estimate Baseline
    log_bg = whittaker_smooth_baseline(log_emp, lam=1e5, p=0.005)
    psd_bg = 10**log_bg
    
    # 2. RAW Excess Log Power (No smoothing applied to this array)
    excess_raw = np.maximum(0, log_emp - log_bg)
    
    # 3. Smoothed Excess Log Power (Generated ONLY for visual comparison in plot)
    sigma_pts = smooth_sigma_hz / df
    excess_smoothed = gaussian_filter1d(excess_raw, sigma=sigma_pts)

    # 4. Index Thresholds
    distance_pts = int(min_dist_hz / df)
    width_pts = int(min_width_hz / df)

    # ==========================================================================
    # CRITICAL CHANGE: RUN ALGORITHM DIRECTLY ON RAW NOISY RESIDUAL (excess_raw)
    # ==========================================================================
    peaks_raw, _ = find_peaks(
        excess_raw,                     # <--- APPLIED ON NON-SMOOTHED SIGNAL
        distance=distance_pts, 
        prominence=min_prominence,
        width=width_pts
    )
    
    prominences_raw, left_bases_raw, right_bases_raw = peak_prominences(excess_raw, peaks_raw)

    extracted_bumps = []
    for i, p_idx in enumerate(peaks_raw):
        l_idx = left_bases_raw[i]
        r_idx = right_bases_raw[i]
        
        extracted_bumps.append({
            'bump_id': i + 1,
            'f0': f_fit[p_idx],
            'f_L': f_fit[l_idx],
            'f_R': f_fit[r_idx],
            'width_hz': f_fit[r_idx] - f_fit[l_idx],
            'p_idx': p_idx,
            'l_idx': l_idx,
            'r_idx': r_idx,
            'prominence': prominences_raw[i]
        })

    return extracted_bumps, psd_bg, excess_raw, excess_smoothed

# ==============================================================================
# 3. RUN ON RAW RESIDUAL
# ==============================================================================

bumps_raw, psd_bg, excess_raw, excess_smoothed = extract_non_parametric_raw(
    f_fit, psd_fit, 
    min_dist_hz=1.5,     # Requires peaks to be at least 1.5 Hz apart
    min_prominence=0.3,  # Ignore minor noise bumps under 0.3 log units
    min_width_hz=0.5,    # Ignore sharp 1-sample spikes
    smooth_sigma_hz=0.3  # Reference smoothing for plot
)

print("\n=========================================================================================")
print(f"{'Bump':<5} | {'Center f0 (Hz)':<15} | {'Base Frequency Interval':<30} | {'Bandwidth (Hz)':<15}")
print("=========================================================================================")
for b in bumps_raw:
    print(f"#{b['bump_id']:<4} | {b['f0']:<15.2f} | "
          f"[{b['f_L']:5.2f} Hz  -  {b['f_R']:5.2f} Hz]                 | "
          f"{b['width_hz']:4.2f} Hz")
print("=========================================================================================\n")

# ==============================================================================
# 4. TWO-PANEL VISUAL DISPLAY
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# --- Panel 1: Empirical PSD & Bumps Found on Raw Residual ---
ax1.semilogy(f_fit, psd_fit, color='gray', alpha=0.5, linewidth=1.2, label='Empirical Welch PSD')
ax1.semilogy(f_fit, psd_bg, 'k--', linewidth=2.0, label='Aperiodic Baseline ($1/f^{\\alpha}$ Background)')

for i, b in enumerate(bumps_raw):
    color = colors[i % len(colors)]
    
    # Shade peak region down to baseline
    mask = (f_fit >= b['f_L']) & (f_fit <= b['f_R'])
    ax1.fill_between(
        f_fit[mask], psd_fit[mask], y2=psd_bg[mask],
        color=color, alpha=0.35,
        label=f"Bump {b['bump_id']} [{b['f_L']:.1f} - {b['f_R']:.1f} Hz]"
    )
    
    # Diamond boundary markers on the baseline
    ax1.plot([b['f_L'], b['f_R']], [psd_bg[b['l_idx']], psd_bg[b['r_idx']]], 
             'D', color=color, markersize=7, markeredgecolor='black')
    
    # Peak center marker
    ax1.plot(b['f0'], psd_fit[b['p_idx']], 'o', color=color, markersize=8, markeredgecolor='black')

ax1.set_ylabel('PSD [Power/Hz]', fontsize=11)
ax1.set_title('Bases Extracted directly on RAW Noisy Residual (No Model Fit & No Pre-smoothing)', fontsize=12)
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc='upper right', framealpha=0.9, fontsize=9)

# --- Panel 2: Raw Noisy vs. Smoothed Excess Power ---
# Plot 1: Raw Noisy Excess Power (Black Curve - Evaluated by Algorithm)
ax2.plot(f_fit, excess_raw, color='black', alpha=0.8, linewidth=1.2, 
         label='RAW Excess Power (Evaluated by find_peaks)')

# Plot 2: Smoothed Curve (Red Dotted - For visual reference)
ax2.plot(f_fit, excess_smoothed, color='red', linestyle=':', alpha=0.7, linewidth=1.8, 
         label='Gaussian Smoothed Reference ($\sigma=0.3$ Hz)')

for i, b in enumerate(bumps_raw):
    color = colors[i % len(colors)]
    
    # Shade excess power region on the raw curve
    mask = (f_fit >= b['f_L']) & (f_fit <= b['f_R'])
    ax2.fill_between(f_fit[mask], excess_raw[mask], y2=0, color=color, alpha=0.35)
    
    # Circle markers for left/right bases directly on the raw signal points
    ax2.plot([b['f_L'], b['f_R']], [excess_raw[b['l_idx']], excess_raw[b['r_idx']]], 
             'o', color=color, markersize=7, markeredgecolor='black')

ax2.set_xlabel('Frequency [Hz]', fontsize=11)
ax2.set_ylabel('Excess Log Power', fontsize=11)
ax2.set_title('Raw Residual Surface with Prominence Bases Extracted Directly on Spikes', fontsize=12)
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend(loc='upper right', framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.show()


Bump  | Center f0 (Hz)  | Base Frequency Interval        | Bandwidth (Hz) 
#1    | 0.56            | [ 0.06 Hz  -   4.00 Hz]                 | 3.94 Hz
#2    | 12.81           | [ 8.19 Hz  -  15.75 Hz]                 | 7.56 Hz
#3    | 23.00           | [22.69 Hz  -  24.50 Hz]                 | 1.81 Hz



## Fit of PSD methods

In [227]:
# ==============================================================================
#  METRIC CALCULATIONS
# ==============================================================================

def compute_metrics(f_grid, psd_candidate, psd_raw):
    log_c = np.log10(psd_candidate)
    log_r = np.log10(psd_raw)
    
    # 1. Accuracy vs Raw PSD
    rmse = np.sqrt(np.mean((log_c - log_r)**2))
    mae = np.mean(np.abs(log_c - log_r))
    
    # 2. Degree of Smoothness (RMS of second derivative)
    d1 = np.gradient(log_c, f_grid)
    d2 = np.gradient(d1, f_grid)
    smoothness_rms = np.sqrt(np.mean(d2**2))
    smoothness_tv = compute_total_variation(f_grid, psd_candidate)
    
    return rmse, mae, smoothness_tv

def compute_total_variation(f_grid, psd_candidate):
    """
    Measures micro-fluctuations (ripples) independently of macro peak height.
    Lower score = Fewer noise ripples (Smoother).
    """
    log_c = np.log10(psd_candidate)
    
    # 1. First Derivative (Slope)
    d1 = np.gradient(log_c, f_grid)
    
    # 2. Total Variation of the Slope (Accumulates penalty for every ripple/directional change)
    slope_changes = np.abs(np.diff(d1))
    tv = np.sum(slope_changes)
    
    return tv

# ==============================================================================
#  RUN EVALUATION & PRINT METRICS TABLE
# ==============================================================================

psd_model = multi_student_t_psd(f_fit, *popt)

psd_sg = smooth_savitzky_golay(f_fit, psd_fit, window_hz=1.0, poly_order=3)
psd_aw = smooth_adaptive_whittaker(psd_fit, lam_base=1e4)
psd_oct = smooth_log_octave(f_fit, psd_fit, fraction=1/12)

curves_to_evaluate = {
    "Empirical Raw Welch PSD": psd_fit,
    f"Student's t Fit (K={best_K})": psd_model,
    "Savitzky-Golay (1.0 Hz)": psd_sg,
    "Adaptive Whittaker": psd_aw,
    "Log-Octave (1/12th)": psd_oct
}

print("\n==========================================================================================================")
print(f"{'Method / Model':<30} | {'Log10 RMSE (vs Raw)':<20} | {'Log10 MAE (vs Raw)':<20} | {'Smoothness (RMS d²y/df²)':<25}")
print("==========================================================================================================")

for name, psd_c in curves_to_evaluate.items():
    rmse, mae, smoothness = compute_metrics(f_fit, psd_c, psd_fit)
    
    rmse_str = f"{rmse:.4f}" if name != "Empirical Raw Welch PSD" else "0.0000 (Reference)"
    mae_str = f"{mae:.4f}" if name != "Empirical Raw Welch PSD" else "0.0000 (Reference)"
    
    print(f"{name:<30} | {rmse_str:<20} | {mae_str:<20} | {smoothness:<25.4f}")

print("==========================================================================================================\n")

# ==============================================================================
# 4. DISPLAY PLOTS (SPECTRA & RESIDUAL ERRORS)
# ==============================================================================

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})

colors = {
    "Empirical Raw Welch PSD": 'gray',
    f"Student's t Fit (K={best_K})": 'red',
    'Savitzky-Golay (1.0 Hz)': '#1f77b4', 
    'Adaptive Whittaker': '#2ca02c', 
    'Log-Octave (1/12th)': '#ff7f0e'
}

# --- Panel 1: Spectral Curves ---
ax1.semilogy(f_fit, psd_fit, color=colors["Empirical Raw Welch PSD"], alpha=0.35, linewidth=1.2, label='Empirical Raw Welch PSD')
ax1.semilogy(f_fit, psd_model, color=colors[f"Student's t Fit (K={best_K})"], linewidth=2.2, label=f"Student's t Fit (K={best_K})")

for name, psd_c in curves_to_evaluate.items():
    if "Empirical" in name or "Student" in name:
        continue
    ax1.semilogy(f_fit, psd_c, linestyle='--', linewidth=1.8, color=colors[name], label=name)

ax1.set_ylabel('PSD [Power/Hz]', fontsize=11)
ax1.set_title('Spectral Curve Approximations', fontsize=12)
ax1.grid(True, which="both", ls="--", alpha=0.5)
ax1.legend(loc='upper right', framealpha=0.9, fontsize=9)

# --- Panel 2: Residual Errors Relative to Raw Empirical PSD ---
log_raw = np.log10(psd_fit)

# Student's t model residual
ax2.plot(f_fit, np.log10(psd_model) - log_raw, color=colors[f"Student's t Fit (K={best_K})"], 
         linewidth=1.8, label=f"Student's t Fit Residual")

# Non-parametric smoother residuals
for name, psd_c in curves_to_evaluate.items():
    if "Empirical" in name or "Student" in name:
        continue
    log_err = np.log10(psd_c) - log_raw
    ax2.plot(f_fit, log_err, linestyle='--', linewidth=1.2, color=colors[name], label=f"{name} Residual")

ax2.axhline(0, color='black', linestyle='--', alpha=0.6)
ax2.set_xlabel('Frequency [Hz]', fontsize=11)
ax2.set_ylabel('Log10 Residual (Curve - Raw)', fontsize=11)
ax2.set_title('Residual Error relative to Raw Welch PSD (log10 Scale)', fontsize=12)
ax2.grid(True, ls="--", alpha=0.5)
ax2.legend(loc='upper right', framealpha=0.9, fontsize=9)

plt.tight_layout()
plt.show()


Method / Model                 | Log10 RMSE (vs Raw)  | Log10 MAE (vs Raw)   | Smoothness (RMS d²y/df²) 
Empirical Raw Welch PSD        | 0.0000 (Reference)   | 0.0000 (Reference)   | 881.5743                 
Student's t Fit (K=4)          | 0.1136               | 0.0908               | 6.8882                   
Savitzky-Golay (1.0 Hz)        | 0.0975               | 0.0781               | 111.1136                 
Adaptive Whittaker             | 0.2438               | 0.2013               | 2.4777                   
Log-Octave (1/12th)            | 0.1111               | 0.0885               | 33.6570                  



In [73]:
file = r'c:\Users\holcman\Documents\GitHub\EEG-labellisation-app---Spectrogram\anesthesia_database\rec_20240321_085300.npy'

y = np.load(file)
y = y[2100 * 128: 2250 * 128]

# 2. Compute Empirical Welch PSD
f_emp, psd_emp = signal.welch(y, fs=128, nperseg=int(128 * 16))

# Restrict fit region up to 30 Hz for accuracy
fit_mask = (f_emp > 0) & (f_emp < 45.0)
f_fit = f_emp[fit_mask]
psd_fit = psd_emp[fit_mask]

# 3. Perform Fit
best_K, N0_est, A_pink_est, alpha_est, components, popt = fit_ou_pink_psd(f_fit, psd_fit, max_components=2)

print("--- FIT RESULTS ---")
print(f"Number of OU Processes (K): {best_K}")
print(f"White Noise Floor (N0): {N0_est:.6f}")
print(f"Pink Noise Amplitude (A_pink): {A_pink_est:.6f}")
print(f"Pink Noise Exponent (alpha): {alpha_est:.3f}")
for comp in components:
    print(comp)

# 4. Plot Results
psd_model = multi_ou_pink_psd(f_fit, popt)

plt.figure(figsize=(9, 5))
plt.semilogy(f_fit, psd_fit, 'b-', alpha=0.6, label='Empirical PSD (Data)')
plt.semilogy(f_fit, psd_model, 'r--', linewidth=2, label=f'Fitted Model (K={best_K} OU + Pink Noise)')
plt.xlabel('Frequency [Hz]')
plt.ylabel('PSD [Power/Hz]')
plt.title('Inverse PSD Fit with Pink Noise & OU Components')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

--- FIT RESULTS ---
Number of OU Processes (K): 2
White Noise Floor (N0): 0.000000
Pink Noise Amplitude (A_pink): 133.484763
Pink Noise Exponent (alpha): 2.500
{'component': 1, 'A_est (c*sigma)': 34.839573541708674, 'lambda_est': 5.608823188147359, 'f0_est_Hz': 12.444317647066558}
{'component': 2, 'A_est (c*sigma)': 9.316573986864558, 'lambda_est': 11.191202473124186, 'f0_est_Hz': 22.37835637571097}


In [172]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define Frequency Vector
f_0 = 10.0          # Center peak frequency (Hz)
fwhm = 2.0          # Full Width at Half Maximum (Hz)
f_max = 50.0        # Range to observe tail evolution
f = np.linspace(0.1, f_max, 2000)

# Peak Height
A_peak = 1.0

# 2. Lorentzian Peak Formulation (OU / Physics model)
# FWHM = 2 * gamma  =>  gamma = fwhm / 2
gamma = fwhm / 2.0
lorentzian = A_peak * (gamma**2 / ((f - f_0)**2 + gamma**2))

# 3. Gaussian Peak Formulation (FOOOF / Statistical model)
# FWHM = 2 * sqrt(2 * ln(2)) * sigma  =>  sigma = fwhm / (2 * sqrt(2 * ln(2)))
sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
gaussian = A_peak * np.exp(-((f - f_0)**2) / (2.0 * sigma**2))

# 4. Plotting Comparison
fig, ax = plt.subplots(figsize=(10, 6))

# Logarithmic Power Scale
ax.semilogy(f, lorentzian, 'r-', linewidth=2.5, label='Lorentzian (OU Heavy Tail: $\sim 1/f^2$)')
ax.semilogy(f, gaussian, 'b--', linewidth=2.5, label='Gaussian (FOOOF Rapid Decay: $\sim e^{-f^2}$)')

# Reference Markings
ax.axvline(f_0, color='gray', linestyle=':', alpha=0.7, label=f'Center Peak ($f_0 = {f_0}$ Hz)')
ax.axhline(A_peak / 2.0, color='black', linestyle='--', alpha=0.4, label='Half-Maximum Power (-3 dB)')

# Annotations
ax.annotate('Matched FWHM region', xy=(f_0 + fwhm/2, A_peak/2), xytext=(f_0 + 3, 0.6),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=6))

ax.annotate('Heavy Lorentzian Tail\n(Decays slowly)', xy=(35, lorentzian[np.argmin(np.abs(f-35))]), 
            xytext=(32, 1e-2),
            arrowprops=dict(facecolor='red', shrink=0.05, width=1, headwidth=6),
            color='red', fontweight='bold')

ax.annotate('Gaussian Queue Plunge\n(Decays exponentially)', xy=(18, gaussian[np.argmin(np.abs(f-18))]), 
            xytext=(20, 1e-6),
            arrowprops=dict(facecolor='blue', shrink=0.05, width=1, headwidth=6),
            color='blue', fontweight='bold')

# Formatting
ax.set_xlim(0, f_max)
ax.set_ylim(1e-12, 2.0)  # Deep log view to expose tails
ax.set_xlabel('Frequency [Hz]', fontsize=12)
ax.set_ylabel('Power Spectral Density [Log Scale]', fontsize=12)
ax.set_title('Tail (Queue) Evolution: Lorentzian vs. Gaussian Peaks', fontsize=14, fontweight='bold')
ax.grid(True, which="both", ls="--", alpha=0.5)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

In [4]:
# --- Parameters
T = 1000 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2]
omega_list = [2*np.pi*1, 2*np.pi*10]
sigma_list = [3, 2]
factor_list = [1, 1]

# --- Generate EEG data
t, y = get_mixed_OU_signals_exact(T, dt, lbda_list, omega_list, sigma_list, factor_list)

t_add, y_add = get_mixed_OU_signals_exact(T, dt, [1], [2*np.pi*30], [2], [1])
y_add += y

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)
spectro_add = spectrogram(y_add, fs, nfft_factor=2)[-1]


# --- Display
fig, axes = plt.subplots(4,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].plot(t, y_add)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].pcolormesh(t_spectro, f_spectro, np.log2(spectro_add + 1e-11), shading = 'nearest', cmap = 'jet')
axes[2].set_title('Spectrogram')
axes[3].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[3].plot(f_spectro, np.log2(np.median(spectro_add, axis = 1)))
axes[3].set_title('PSD')

axes[1].sharex(axes[0])


plt.show()


